In [1]:
%%capture
!pip install -U openai pydantic langchain langchain-community langchain-openai datasets fsspec faiss-gpu sentence_transformers huggingface-hub faiss-gpu

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter Your OpenAI API Key:")

Enter Your OpenAI API Key:··········


In [3]:
import os
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("Enter Your HUggingFace API Key:")

Enter Your HUggingFace API Key:··········


In [4]:
from langchain.globals import set_verbose

set_verbose(True)

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#🔗 **Origins of CoT Prompting**

- CoT was introduced by Wei et al. in their 2022 paper, [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://browse.arxiv.org/pdf/2201.11903.pdf).

- It's a response to the need for better reasoning in large language models.

💡 **What is CoT Prompting?**
- CoT stands for Chain of Thought Prompting.

- It's about guiding language models through a step-by-step reasoning process.

- The model shows its reasoning, forming a "chain" of thoughts, rather than just giving an answer.

- Mimics human-like thought processes for complex tasks.


### 🎓 **Standard vs. CoT Prompting Example**

<figure>
  <img src="https://deepgram.com/_next/image?url=https%3A%2F%2Fwww.datocms-assets.com%2F96965%2F1696369825-cot-example.png&w=3840&q=75" alt="Image Description" style="width:100%">
  <figcaption>Examples of ⟨input, chain of thought, output⟩ triples for arithmetic, commonsense, and symbolic reasoning benchmarks. Chains of thought are highlighted.</figcaption>
  <a href="https://deepgram.com/_next/image?url=https%3A%2F%2Fwww.datocms-assets.com%2F96965%2F1696369825-cot-example.png&w=3840&q=75">Image Source</a>
</figure>

- Standard: Direct answer with no reasoning (e.g., "11").

- CoT: Detailed reasoning steps (e.g., "Roger started with 5, added 6 from 2 cans, so 5 + 6 = 11").

### 🚀 **Usage of CoT Prompting**

1. **Enhanced Reasoning**: Breaks down complex problems into simpler steps.

2. **Combination with Few-Shot Prompting**: Uses examples to guide model responses, great for complex tasks.

3. **Interpretability**: Offers a clear view of the model's thought process.

4. **Applications**: Useful in arithmetic, commonsense reasoning, and symbolic tasks.

### ✨ **Impact of CoT Prompting**

- Improves accuracy and insightfulness in responses.

- Focuses on reasoning and interpretability, especially in complex scenarios.

The code below downloads an dataset which has over [1.8 million Chain of Thought examples](https://huggingface.co/datasets/kaist-ai/CoT-Collection).

In [14]:
import json
import pandas as pd

file_path = "/content/drive/MyDrive/CoT_collection_en.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert the dictionary of dicts into a list of dicts
data_list = list(data.values())

# Now convert to DataFrame
df = pd.DataFrame(data_list)

# Check columns
print(df.columns)

# If you're sure these columns exist
df = df[['source', 'rationale', 'target']].dropna()

# Optional: Sample if more than 1000
df = df.sample(n=min(1000, len(df)), random_state=42).reset_index(drop=True)

# Convert to list of dicts for LangChain
selected_examples = df.to_dict(orient="records")

Index(['source', 'target', 'rationale', 'config', 'task', 'prompt'], dtype='object')


We'll sample just 1000 of these Chain of Thought examples and save them as a list of dictionaries.

In [15]:
selected_examples[0]

{'source': 'Two analogies that relate actions with their consequences are given in the form "A : B. C : ?". The phrase "A : B" relates action A to consequence B. Your task is to replace the question mark (?) with the appropriate consquence of the given action C, following the "A : B" relation. Your answer should be a single verb, without further explanation.\n\ntravel : arrive. cut : ?',
 'rationale': 'Traveling to a destination typically results in arriving at that destination. Similarly, cutting something will result in bleeding.',
 'target': 'bleed'}

For this example we'll use an open source embedding model from HuggingFace so we don't accumulate a huge bill for OpenAI

In [16]:
!pip install -U langchain-community
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

model_name = "BAAI/bge-base-en-v1.5"

encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity

embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cpu'},
    encode_kwargs=encode_kwargs
)

To set up a Chain of Thought prompt you will use a `FewShotPromptTemplate` as well as an example selector.

In this example you'll use `MaxMarginalRelevanceExampleSelector`, but you can use any of the example selectors you learned about before.

In [17]:
from langchain.prompts.example_selector import MaxMarginalRelevanceExampleSelector

from langchain.prompts import FewShotPromptTemplate, PromptTemplate

from langchain_community.vectorstores import FAISS

In [18]:
prefix = "Consider the following as examples of how to reason:"

examples_template = """Query: {source}

Rationale: {rationale}

Response: {target}
"""

suffix = """Using a similar reasoning approach, answer the users question which is delimited by triple backticks.

User question: ```{input}```

Take a deep breath, break down the user's query step-by-step, and provide a clear chain of thought in your response."
"""

In [19]:
examples_prompt = PromptTemplate(
    input_variables=["source", "rationale", "target"],
    template=examples_template
)

There's a large number of examples to embed and add to the vector store. The below cell will take a few minutes to run. It took me ~3 minutes using the free T4 GPU provided on Google colab.

In [20]:
!pip install faiss-cpu
example_selector = MaxMarginalRelevanceExampleSelector.from_examples(
    # The list of examples available to select from.
    selected_examples,
    # The embedding class used to produce embeddings which are used to measure semantic similarity.
    embeddings,
    # The VectorStore class that is used to store the embeddings and do a similarity search over.
    FAISS,
    # The number of examples to produce.
    k=7,
)

In [21]:
mmr_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=examples_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["input"]
)

In [22]:
query = """Here's a short story: A lifter uses a barbell, but moves a jump rope\
in a wider arc. The object likely to travel further is (A) the jump rope (B) the\
barbell.

What is the most sensical answer between "a jump rope" and "a barbell"?
"""

In [23]:
print(mmr_prompt.format(input=query))

Consider the following as examples of how to reason:

Query: Here's a short story: Ben was at the playground. He noticed when he rolled his ball across the sand that it moved slower than when he rolled it across a wooden plank. This is because the _____ has less resistance. (A) wooden plank (B) sand.

What is the most sensical answer between "sand" and  "wooden plank"?

Rationale: The fact that the ball moves slower on sand is due to its greater resistance. The answer, therefore, is "sand".

Response: sand


Query: Single/multi-select question: If "A worker suspended in the air.", can we conclude "A man is sky diving."?

OPTIONS:
- yes
- it is not possible to tell
- no

Rationale: The premise describes a worker suspended in the air, and it is likely through means of equipment or machinery. In contrast, skydiving involves an individual jumping from high altitudes with no support from any other source. So we cannot conclude that he is sky diving just because he is suspended in mid-air. H

In [26]:
!pip install -U langchain-openai
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4-0125-preview", temperature=0.0)

llm_chain = mmr_prompt | llm | StrOutputParser()

for chunk in llm_chain.stream({"input":query}):
  print(chunk, end="", flush=True)

  Using cached langchain_openai-0.3.28-py3-none-any.whl.metadata (2.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 851.1 kB/s eta 0:00:00
To determine the most sensical answer between "a jump rope" and "a barbell" in terms of which object is likely to travel further, let's analyze the information provided in the short story:

1. **Understanding the Objects**: The story mentions two objects, a barbell and a jump rope. A barbell is typically used for weightlifting and is not designed to travel far distances. It is heavy and meant to be lifted, not thrown or propelled. On the other hand, a jump rope is light and designed to be swung in an arc around the body. 

2. **Movement Described**: The lifter uses a barbell, which implies lifting movements rather than throwing or propelling movements. For the jump rope, it is mentioned that it is moved in a wider arc. This suggests that the jump rope is being swung, which is its intended use.

3. **Physics Consideration**: When consi

In [27]:
query = """
Given the fact that: Inhaling, or breathing in, increases the size of the chest, \
 which decreases air pressure inside the lungs. Answer the question: If Mona is \
 done with a race and her chest contracts, what happens to the amount of air \
 pressure in her lungs increases or decreases?
"""

In [28]:
for chunk in llm_chain.stream({"input":query}):
  print(chunk, end="", flush=True)

Given the fact that inhaling, or breathing in, increases the size of the chest, which decreases air pressure inside the lungs, we can infer a basic principle of how air pressure in the lungs is related to the size of the chest cavity. When the chest expands, air pressure inside the lungs decreases, allowing air to flow in. Conversely, when the chest contracts, the volume of the chest cavity decreases, which must increase the air pressure inside the lungs because the same amount of air is now compressed into a smaller space.

So, if Mona has just finished a race and her chest contracts (which means her chest is decreasing in size as she exhales), the amount of air pressure in her lungs increases. This is because the contraction of the chest decreases the volume available for the air inside the lungs, thereby increasing the pressure.

Response: increases